# 02 - Preprocessamento e modelo

Objetivo: carregar reviews, tokenizar/vetorizar os textos e treinar um modelo neural para classificar sentimento.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "aclImdb"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

print("Projeto:", PROJECT_ROOT)
print("Dados:", DATA_DIR)


Projeto: C:\Users\Carlos henrique\Documents\deeplearning
Dados: C:\Users\Carlos henrique\Documents\deeplearning\data\raw\aclImdb


In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from src.imdb_data import load_reviews

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


def clean_text_array(values):
    cleaned = []
    for text in values:
        text = text.replace("<br />", " ")
        text = text.encode("ascii", "ignore").decode("ascii")
        cleaned.append(text)
    return np.array(cleaned, dtype=object)


TensorFlow: 2.21.0


## Carregamento e divisao

In [3]:
df = load_reviews(DATA_DIR)
train_df = df[df["split"] == "train"].sample(frac=1, random_state=42).reset_index(drop=True)
test_df = df[df["split"] == "test"].sample(frac=1, random_state=42).reset_index(drop=True)

x_train = clean_text_array(train_df["review"].to_numpy())
y_train = train_df["target"].to_numpy()
x_test = clean_text_array(test_df["review"].to_numpy())
y_test = test_df["target"].to_numpy()

print("Treino:", x_train.shape, y_train.shape)
print("Teste:", x_test.shape, y_test.shape)
train_df[["split", "label", "target", "review"]].head()


Treino: (25000,) (25000,)
Teste: (25000,) (25000,)


,split,label,target,review
0,train,neg,0,"Silent Night, Deadly Night 5 is the very last ..."
1,train,pos,1,The idea ia a very short film with a lot of in...
2,train,neg,0,"For me, this movie just seemed to fall on its ..."
3,train,pos,1,Was this based on a comic-book? A video-game? ...
4,train,pos,1,Caution: May contain spoilers...<br /><br />I'...


## Vetorizacao dos textos

In [4]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 250

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(x_train)

vocab = vectorizer.get_vocabulary()
print("Tamanho do vocabulario:", len(vocab))
print("Primeiros tokens:", vocab[:20])


Tamanho do vocabulario: 20000
Primeiros tokens: ['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('was'), np.str_('as'), np.str_('for'), np.str_('with'), np.str_('movie'), np.str_('but'), np.str_('film')]


In [5]:
example_text = x_train[0]
print(example_text[:500])
print(vectorizer([example_text]).numpy()[0][:40])


Silent Night, Deadly Night 5 is the very last of the series, and like part 4, it's unrelated to the first three except by title and the fact that it's a Christmas-themed horror flick.  Except to the oblivious, there's some obvious things going on here...Mickey Rooney plays a toymaker named Joe Petto and his creepy son's name is Pino. Ring a bell, anyone? Now, a little boy named Derek heard a knock at the door one evening, and opened it to find a present on the doorstep for him. Even though it sa
[1252  310 2440  310  664    7    2   52  227    5    2  200    3   38
  170  653   29 5676    6    2   85  291  546   32  416    3    2  186
   12   29    4    1  196  503  546    6    2 7838  216   46]


## Modelo neural

In [6]:
inputs = keras.Input(shape=(1,), dtype=tf.string, name="review")
x = vectorizer(inputs)
x = layers.Embedding(input_dim=len(vocab), output_dim=64, mask_zero=True, name="embedding")(x)
x = layers.Bidirectional(layers.LSTM(32), name="bilstm")(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(32, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid", name="sentiment")(x)

model = keras.Model(inputs, outputs, name="imdb_sentiment_bilstm")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()


Model: "imdb_sentiment_bilstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ review (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 250)       │          0 │ review[0][0]      │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 250, 64)   │  1,280,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 250)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, 64)        │     24,832 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ bilstm[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sentiment (Dense)   │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,306,945 (4.99 MB)

 Trainable params: 1,306,945 (4.99 MB)

 Non-trainable params: 0 (0.00 B)

## Treinamento

In [7]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True),
]

history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/3


 1/79 ━━━━━━━━━━━━━━━━━━━━ 4:56 4s/step - accuracy: 0.4766 - loss: 0.6936

 2/79 ━━━━━━━━━━━━━━━━━━━━ 34s 454ms/step - accuracy: 0.4775 - loss: 0.6935

 3/79 ━━━━━━━━━━━━━━━━━━━━ 34s 451ms/step - accuracy: 0.4798 - loss: 0.6935

 4/79 ━━━━━━━━━━━━━━━━━━━━ 33s 447ms/step - accuracy: 0.4851 - loss: 0.6934

...

76/79 ━━━━━━━━━━━━━━━━━━━━ 1s 482ms/step - accuracy: 0.9356 - loss: 0.1862

77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step - accuracy: 0.9356 - loss: 0.1860

78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 484ms/step - accuracy: 0.9357 - loss: 0.1858

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step - accuracy: 0.9358 - loss: 0.1856

79/79 ━━━━━━━━━━━━━━━━━━━━ 39s 495ms/step - accuracy: 0.9420 - loss: 0.1708 - val_accuracy: 0.8394 - val_loss: 0.4071


In [8]:
history_df = pd.DataFrame(history.history)
history_df


,accuracy,loss,val_accuracy,val_loss
0,0.66580,0.593783,0.8518,0.380405
1,0.89075,0.287110,0.8718,0.316051
2,0.94205,0.170846,0.8394,0.407071


## Avaliacao rapida e salvamento

In [9]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
print(f"Loss teste: {test_loss:.4f}")
print(f"Accuracy teste: {test_accuracy:.4f}")

model_path = MODELS_DIR / "imdb_sentiment_model.keras"
history_path = MODELS_DIR / "training_history.csv"

model.save(model_path)
history_df.to_csv(history_path, index=False)

print("Modelo salvo em:", model_path)
print("Historico salvo em:", history_path)


Loss teste: 0.3614
Accuracy teste: 0.8477
Modelo salvo em: C:\Users\Carlos henrique\Documents\deeplearning\models\imdb_sentiment_model.keras
Historico salvo em: C:\Users\Carlos henrique\Documents\deeplearning\models\training_history.csv
